In [212]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [213]:
df = sns.load_dataset('titanic')

In [214]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [215]:
df['age'] = pd.to_numeric(df['age'],errors='coerce').round().astype('Int32')

In [216]:
df['family'] = df.sibsp + df.parch

In [217]:
df.sample(4)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family
749,0,3,male,31,0,0,7.75,Q,Third,man,True,NaN,Queenstown,no,True,0
616,0,3,male,34,1,1,14.40,S,Third,man,True,NaN,Southampton,no,False,2
322,1,2,female,30,0,0,12.35,Q,Second,woman,False,NaN,Queenstown,yes,True,0
822,0,1,male,38,0,0,0.00,S,First,man,True,NaN,Southampton,no,True,0


In [218]:
x = df.drop(columns=['survived','sibsp','parch','who','deck','alive','alone','embarked'],axis=1)
y = df['survived']

In [219]:
x.head()

,pclass,sex,age,fare,class,adult_male,embark_town,family
0,3,male,22,7.2500,Third,True,Southampton,1
1,1,female,38,71.2833,First,False,Cherbourg,1
2,3,female,26,7.9250,Third,False,Southampton,0
3,1,female,35,53.1000,First,False,Southampton,1
4,3,male,35,8.0500,Third,True,Southampton,0


In [220]:
from sklearn.model_selection import  train_test_split
from sklearn.preprocessing import  OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier

In [221]:
x_train, x_test, y_train, y_test = train_test_split(x, y, shuffle=True, test_size=0.20, random_state=42)

In [222]:
x_train

,pclass,sex,age,fare,class,adult_male,embark_town,family
331,1,male,46,28.5000,First,True,Southampton,0
733,2,male,23,13.0000,Second,True,Southampton,0
382,3,male,32,7.9250,Third,True,Southampton,0
704,3,male,26,7.8542,Third,True,Southampton,1
813,3,female,6,31.2750,Third,False,Southampton,6
...,...,...,...,...,...,...,...,...
106,3,female,21,7.6500,Third,False,Southampton,0
270,1,male,<NA>,31.0000,First,True,Southampton,0
860,3,male,41,14.1083,Third,True,Southampton,2
435,1,female,14,120.0000,First,False,Southampton,3


In [223]:
x_test

,pclass,sex,age,fare,class,adult_male,embark_town,family
709,3,male,<NA>,15.2458,Third,True,Cherbourg,2
439,2,male,31,10.5000,Second,True,Southampton,0
840,3,male,20,7.9250,Third,True,Southampton,0
720,2,female,6,33.0000,Second,False,Southampton,1
39,3,female,14,11.2417,Third,False,Cherbourg,1
...,...,...,...,...,...,...,...,...
433,3,male,17,7.1250,Third,True,Southampton,0
773,3,male,<NA>,7.2250,Third,True,Cherbourg,0
25,3,female,38,31.3875,Third,False,Southampton,6
84,2,female,17,10.5000,Second,False,Southampton,0


In [224]:
y_train

331    0
733    0
382    0
704    0
813    0
      ..
106    1
270    0
860    0
435    1
102    0
Name: survived, Length: 712, dtype: int64

In [225]:
y_test

709    1
439    0
840    0
720    1
39     1
      ..
433    0
773    0
25     1
84     1
10     1
Name: survived, Length: 179, dtype: int64

In [226]:
x_train

,pclass,sex,age,fare,class,adult_male,embark_town,family
331,1,male,46,28.5000,First,True,Southampton,0
733,2,male,23,13.0000,Second,True,Southampton,0
382,3,male,32,7.9250,Third,True,Southampton,0
704,3,male,26,7.8542,Third,True,Southampton,1
813,3,female,6,31.2750,Third,False,Southampton,6
...,...,...,...,...,...,...,...,...
106,3,female,21,7.6500,Third,False,Southampton,0
270,1,male,<NA>,31.0000,First,True,Southampton,0
860,3,male,41,14.1083,Third,True,Southampton,2
435,1,female,14,120.0000,First,False,Southampton,3


In [227]:
categorical_col = ['sex','class','adult_male','embark_town']
numerical_col = ['pclass','age','fare','family']

In [228]:
categorical_trns = Pipeline(steps=[
    ('impute',SimpleImputer(strategy='most_frequent',add_indicator=True)),
    ('ohe',OneHotEncoder(handle_unknown='ignore'))
])

In [229]:
numerical_trns = Pipeline(steps=[
    ('impute',IterativeImputer()),
    ('scaler',StandardScaler())
])

In [230]:
colwise_trns = ColumnTransformer(transformers=[
    ('cotegorical',categorical_trns,categorical_col),
    ('numerical',numerical_trns,numerical_col)
])

In [231]:
model_pipline = Pipeline(steps=[
    ('transform',colwise_trns),
    ('pca',PCA(n_components=6)),
    ('model',LogisticRegressionCV())
]
)

In [232]:
model_pipline.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('transform', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cotegorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different t

In [233]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

In [234]:
y_pred = model_pipline.predict(x_test)

In [235]:
print(f'accuracy is {accuracy_score(y_test,y_pred)}')

accuracy is 0.8044692737430168


In [236]:
print(confusion_matrix(y_test,y_pred))

[[91 14]
 [21 53]]


In [237]:
x_temp = model_pipline.named_steps['transform'].transform(x)
x_trasformed = model_pipline.named_steps['pca'].transform(x_temp)

In [238]:
x_trasformed

array([[ 1.24438464, -0.28832488,  0.62126397, -0.077517  , -0.0570175 ,
         0.20027248],
       [-2.20271986,  0.68773714, -0.96662998, -0.14083234,  0.16558058,
        -0.9643706 ],
       [ 0.92932993,  0.10079834, -1.31924698, -0.2594617 ,  0.47924692,
         0.49091125],
       ...,
       [ 1.14146484,  1.60592858, -0.57429988,  0.48691567,  0.39320628,
         0.22622908],
       [-1.37397866, -0.46652111,  0.37444533, -0.77490094, -1.02687115,
        -1.23372502],
       [ 0.89498038, -1.01710487,  0.27223715, -0.31876545,  0.56472615,
        -0.5559987 ]], shape=(891, 6))

In [239]:
import plotly.express as px

In [240]:
columns = [f'pca{i}' for i in range(1,x_trasformed.shape[1]+1)]

In [241]:
pca_data = pd.DataFrame(x_trasformed,columns=columns)

In [242]:
pc = model_pipline.named_steps['pca']

In [243]:
fig = px.line(
    x=range(1,len(pc.explained_variance_ratio_)+1),
    y=pc.explained_variance_ratio_.cumsum(),
    markers = True,
    title='cumilative expalined variance',
    labels={'x':'componets','y':'cumilative veriacne'}
)
fig.add_hline(y=0.95,line_dash='dash',line_color='pink',annotation_text='90% treshold')
fig.show()